# Lab 2: RAG for Food Safety Knowledge Support
This version uses more specific PDF documents from Thai food/agriculture agencies:


- ACFS/TAS: Good practices for ready-to-eat fresh pre-cut fruits and vegetables
- ACFS/TAS: Good practices for fresh fruit and vegetable packing houses
- ACFS: Manual for registering food producers exporting to China under Decree 248

**Main task:** help QA / regulatory / operations learners answer detailed questions from a controlled document set.

**Why this corpus works better for RAG:** the documents contain specific scopes, procedures, evidence requirements, product categories, and registration paths. A model answering without RAG will usually give generic food-safety advice, while RAG can cite document-specific requirements.

**Safety note:** this lab is for training only. RAG can help find relevant guidance and draft evidence-based summaries, but it must not decide product release, hold, quarantine, recall scope, CCP disposition, export eligibility, or regulatory approval.

**Demo flow**

1. Download specific food-domain PDFs with `wget`
2. Load PDF pages as LangChain documents
3. Split pages into chunks
4. Embed and store chunks in a vector database
5. Retrieve context for specific food-domain questions
6. Compare answer without RAG vs with RAG

## 0. Setup: choose OpenAI or Groq

The reference notebook supports multiple providers. This lab keeps the same idea and supports:

- `ChatOpenAI`
- `ChatGroq`

For embeddings, we use a local HuggingFace embedding model. This is separate from the chat model.

In [ ]:
# %%capture
# !pip install -qU langchain-openai

In [ ]:
# if using groq api (Free Tier) => don't run the cell above, run this block instead
!pip install -qU langchain-groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 6.5 MB/s eta 0:00:00


Run these required libraries.

In [ ]:
%%capture
!pip install -qU pypdf # pdf loader
!pip install -q fixthaipdf # pdf loader
!pip install -qU langchain # main framework
!pip install -qU langchain-community # chunking tools
!pip install -qU langchain-huggingface # embedding models
!pip install -qU langchain-chroma # vector database

# !pip install sentence-transformers

In [ ]:
import os
import json
import getpass
import subprocess
from pathlib import Path


PROVIDER = "groq"  # Change only this line:

if PROVIDER == "openai":
    from langchain_openai import ChatOpenAI
    if "OPENAI_API_KEY" not in os.environ:
        os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API key: ")
    llm = ChatOpenAI(
        model="gpt-5-mini",
        temperature=0,
        max_retries=2,
    )
elif PROVIDER == "groq":
    from langchain_groq import ChatGroq
    if "GROQ_API_KEY" not in os.environ:
        os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your Groq API key: ")
    llm = ChatGroq(
        model="llama-3.3-70b-versatile", # update
        temperature=0,
        max_tokens=2000,
        timeout=None,
        max_retries=2,
    )
else:
    raise ValueError("PROVIDER must be 'openai' or 'groq'")

print(f"Using provider: {PROVIDER}")

Enter your Groq API key: ··········
Using provider: groq


## 1. Download Food-domain PDFs with `wget`

**Task:** download the reference documents that the RAG system is allowed to use.

We use `wget` so the notebook is self-contained: learners can rerun the notebook and recreate the local PDF corpus.

In [ ]:
PDF_DIR = Path("./rag_pdfs")
PDF_DIR.mkdir(parents=True, exist_ok=True)

pdf_sources = [
    {
        "doc_id": "tas_9039_fresh_cut_rte",
        "title": "TAS 9039-2556: Good Manufacturing Practices for Ready-to-Eat Fresh Pre-Cut Fruits and Vegetables",
        "url": "https://tas2go.acfs.go.th/upload_standard/43_th.pdf",
        "filename": "tas_9039_fresh_cut_rte.pdf",
        "topic": "fresh-cut ready-to-eat produce",
    },
    {
        "doc_id": "tas_9035_packing_house",
        "title": "TAS 9035-2563: Good Practices for Fresh Fruit and Vegetable Packing House",
        "url": "https://tas2go.acfs.go.th/upload_standard/412_th.pdf",
        "filename": "tas_9035_packing_house.pdf",
        "topic": "fresh produce packing house",
    },
    {
        "doc_id": "acfs_decree_248_china_export",
        "title": "ACFS Manual: Food Producer Registration for Export to China under Decree 248",
        "url": "https://e-book.acfs.go.th/backend/uploads/Download/8cf95fc81b78906756467afbd81974c6.pdf",
        "filename": "acfs_decree_248_china_export.pdf",
        "topic": "China export registration",
    },
]

for item in pdf_sources:
    out_path = PDF_DIR / item["filename"]
    if out_path.exists() and out_path.stat().st_size > 0:
        print(f"Already exists: {out_path}")
        continue

    print(f"Downloading: {item['title']}")
    subprocess.run(
        ["wget", "-O", str(out_path), item["url"]],
        check=True,
    )

print("\nDownloaded / available PDFs:")
for item in pdf_sources:
    path = PDF_DIR / item["filename"]
    print("-", path, "| bytes:", path.stat().st_size if path.exists() else "missing")

Downloading: TAS 9039-2556: Good Manufacturing Practices for Ready-to-Eat Fresh Pre-Cut Fruits and Vegetables
Downloading: TAS 9035-2563: Good Practices for Fresh Fruit and Vegetable Packing House
Downloading: ACFS Manual: Food Producer Registration for Export to China under Decree 248

Downloaded / available PDFs:
- rag_pdfs/tas_9039_fresh_cut_rte.pdf | bytes: 820901
- rag_pdfs/tas_9035_packing_house.pdf | bytes: 840888
- rag_pdfs/acfs_decree_248_china_export.pdf | bytes: 6973835


## 2. Load PDF Pages as Documents

**Task:** load each PDF page as a LangChain document.

The original reference notebook uses `PyPDFLoader` for a PDF. We keep that pattern, but use food-domain PDFs instead.

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from fixthaipdf import clean
pages = []

for item in pdf_sources:
    path = PDF_DIR / item["filename"]
    loader = PyPDFLoader(str(path))
    loaded_pages = loader.load()

    for page in loaded_pages:
        page.page_content = clean(page.page_content)
        page.metadata.update({
            "doc_id": item["doc_id"],
            "title": item["title"],
            "url": item["url"],
            "topic": item["topic"],
            "source_path": str(path),
        })
    pages.extend(loaded_pages)

print(f"Loaded {len(pages)} PDF pages")
print("Example page metadata:", pages[0].metadata)
print("\nExample page preview:")
print(pages[0].page_content[:900])

/tmp/ipykernel_1838/2397288530.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Loaded 91 PDF pages
Example page metadata: {'producer': 'Adobe Acrobat Pro 9.0.0', 'creator': 'Adobe Acrobat Pro 9.0.0', 'creationdate': '2013-12-04T14:21:13+07:00', 'author': 'rmt1', 'moddate': '2014-03-29T16:11:42+07:00', 'title': 'TAS 9039-2556: Good Manufacturing Practices for Ready-to-Eat Fresh Pre-Cut Fruits and Vegetables', 'source': 'rag_pdfs/tas_9039_fresh_cut_rte.pdf', 'total_pages': 16, 'page': 0, 'page_label': '1', 'doc_id': 'tas_9039_fresh_cut_rte', 'url': 'https://tas2go.acfs.go.th/upload_standard/43_th.pdf', 'topic': 'fresh-cut ready-to-eat produce', 'source_path': 'rag_pdfs/tas_9039_fresh_cut_rte.pdf'}

Example page preview:
มาตรฐานสินค้าเกษตร
มกษ. 9039-2556
THAI AGRICULTURAL STANDARD
TAS 9039-2013
การปฏิบัติที่ดีสำหรับการผลิตผักและผลไม้สดตัดแต่งพร้อมบริโภค
GOOD MANUFACTURING PRACTICES FOR
READY-TO-EAT FRESH PRE-CUT FRUITS AND
VEGETABLES
สำนักงานมาตรฐานสินค้าเกษตรและอาหารแห่งชาติ
กระทรวงเกษตรและสหกรณ์
ICS 67.020 ISBN 978-74-403-658-2


## 3. Split Pages into Chunks

**Task:** break PDF page text into smaller chunks for retrieval.

RAG usually does not send a whole PDF to the model. It retrieves only the chunks most relevant to the question.

**RAG concept:** chunk size and overlap affect retrieval quality.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=900,
    chunk_overlap=180,
    add_start_index=True,
)

splits = text_splitter.split_documents(pages)

for i, doc in enumerate(splits):
    doc.metadata["chunk_id"] = i

print(f"Split {len(pages)} pages into {len(splits)} chunks")
print("Example chunk metadata:", splits[0].metadata)
print("\nExample chunk preview:")
print(splits[0].page_content[:700])

Split 91 pages into 176 chunks
Example chunk metadata: {'producer': 'Adobe Acrobat Pro 9.0.0', 'creator': 'Adobe Acrobat Pro 9.0.0', 'creationdate': '2013-12-04T14:21:13+07:00', 'author': 'rmt1', 'moddate': '2014-03-29T16:11:42+07:00', 'title': 'TAS 9039-2556: Good Manufacturing Practices for Ready-to-Eat Fresh Pre-Cut Fruits and Vegetables', 'source': 'rag_pdfs/tas_9039_fresh_cut_rte.pdf', 'total_pages': 16, 'page': 0, 'page_label': '1', 'doc_id': 'tas_9039_fresh_cut_rte', 'url': 'https://tas2go.acfs.go.th/upload_standard/43_th.pdf', 'topic': 'fresh-cut ready-to-eat produce', 'source_path': 'rag_pdfs/tas_9039_fresh_cut_rte.pdf', 'start_index': 0, 'chunk_id': 0}

Example chunk preview:
มาตรฐานสินค้าเกษตร
มกษ. 9039-2556
THAI AGRICULTURAL STANDARD
TAS 9039-2013
การปฏิบัติที่ดีสำหรับการผลิตผักและผลไม้สดตัดแต่งพร้อมบริโภค
GOOD MANUFACTURING PRACTICES FOR
READY-TO-EAT FRESH PRE-CUT FRUITS AND
VEGETABLES
สำนักงานมาตรฐานสินค้าเกษตรและอาหารแห่งชาติ
กระทรวงเกษตรและสหกรณ์
ICS 67.020 ISBN 978-74-

## 4. Embed and Store Chunks in a Vector Database

**Task:** convert chunks into embeddings and store them for similarity search.

This follows the same structure as the reference notebook:

`PDF pages -> chunks -> embeddings -> vector store`

In [ ]:
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

# bge-m3 works better for multilingual Thai/English retrieval, but it is larger than bge-small.
embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-m3")

vector_store = Chroma(
    collection_name="food_safety_specific_pdf_rag_demo",
    embedding_function=embeddings,
)

document_ids = vector_store.add_documents(documents=splits)
print(f"Stored {len(document_ids)} chunks in vector store")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

Stored 176 chunks in vector store


## 5. Retrieve Context for Specific Food-domain Questions

**Task:** search the PDF corpus for chunks relevant to a detailed operational/regulatory question.

The query examples are intentionally specific. This makes the difference between no-RAG and RAG easier to see.

In [ ]:
test_queries = {
    "fresh_cut_label_storage":
        "สำหรับผักและผลไม้สดตัดแต่งพร้อมบริโภค ควรมีข้อมูลผลิตภัณฑ์หรือข้อมูลฉลากอะไรบ้างเกี่ยวกับการเก็บรักษา การใช้งาน การระบุวันที่ อุณหภูมิ และสารก่อภูมิแพ้?",

    "fresh_cut_temperature_control":
        "ในการผลิตผักและผลไม้สดตัดแต่งพร้อมบริโภค มีข้อกำหนดอะไรบ้างเกี่ยวกับการควบคุมอุณหภูมิ การเก็บรักษา และการขนส่ง?",

    "packing_house_receiving_control":
        "โรงคัดบรรจุผักและผลไม้สดควรมีการควบคุมหรือบันทึกอะไรบ้างในการรับวัตถุดิบหรือผลิตผลเข้ามาในโรงคัดบรรจุ?",

    "packing_house_cross_contamination":
        "โรงคัดบรรจุผักและผลไม้สดควรออกแบบพื้นที่ เครื่องมือ และการปฏิบัติงานอย่างไรเพื่อลดการปนเปื้อนและการปนเปื้อนข้าม?",

    "china_decree_248_registration":
        "อาหารที่ควบคุมในการส่งออกไปจีน มีกี่อย่าง? อะไรบ้าง?",
}

### Test with this question : "china_decree_248_registration"

In [ ]:
query_key = "china_decree_248_registration"
# can be changed to "packing_house_uncertified_supplier" or "china_decree_248_registration" for experimentation.


In [ ]:
query = test_queries[query_key]

retrieved_docs = vector_store.similarity_search(query, k=5)

print(f"Query key: {query_key}")
print(f"Query: {query}\n")
print("Retrieved Documents")
print("=" * 80)

for rank, doc in enumerate(retrieved_docs, start=1):
    print(f"[Source {rank}] {doc.metadata['title']}")
    print("URL:", doc.metadata["url"])
    print("Page:", doc.metadata.get("page"), "| chunk:", doc.metadata.get("chunk_id"))
    print(doc.page_content[:1000].replace("\n", " "))
    print("-" * 80)

Query key: china_decree_248_registration
Query: อาหารที่ควบคุมในการส่งออกไปจีน มีกี่อย่าง? อะไรบ้าง?

Retrieved Documents
[Source 1] ACFS Manual: Food Producer Registration for Export to China under Decree 248
URL: https://e-book.acfs.go.th/backend/uploads/Download/8cf95fc81b78906756467afbd81974c6.pdf
Page: 4 | chunk: 102
ผู้ผลิตก่อน จึงจะสามารถส่งออกไปจีนได้ โดยมีผลบังคับใช้ตั้งแต่วันที่ 1 มกราคม 2565 เป็นต้นมา โดยระเบียบดังกล่าวนำมาแทนที่กฎระเบียบ Decree 145 ว่าด้วยการขึ้นทะเบียน ผู้ผลิตอาหาร ในต่างประเทศของสำนักงานควบคุมคุณภาพ ตรวจสอบและกักกันโรคแห่งประเทศจีน (The General Administration of Quality Supervision, Inspection and Quarantine: AQSIQ Decree 145) ที่เดิมระบุ เฉพาะ สินค้าเสี่ยงสูง บางรายการ (เนื้อสัตว์ ผลิตภัณฑ์นม สินค้าประมง และรังนก) ต้องขึ้นทะเบียนดังกล่าว ในการขึ้นทะเบียนสำนักงานศุลกากรจีน (GACC) จะให้ดำเนินการผ่านระบบที่เรียกว่า “CIFER” ซึ่งจะ ดำเนินการผ่าน... CIFER.SINGLEWINDOW.CN
--------------------------------------------------------------------------------
[Source 2

## 6. Compare Without RAG vs With RAG

**Task:** answer the same question in two ways:

- **Without RAG:** model answers from general knowledge.
- **With RAG:** model must answer using retrieved PDF context.

This is the main teaching moment. With a specific PDF corpus, the RAG answer should cite document-specific details that a generic answer may miss.

In [ ]:
system_prompt = """
You are a QA and regulatory knowledge-support assistant for food manufacturing.
You may summarize reference material and suggest evidence or records to review.
If source context is missing or insufficient, say what is not found in the provided context.
Keep facts, interpretations, and human approval needs separate.
""".strip()

no_rag_prompt = f"""
Answer this food QA / regulatory question.

Question:
{query}
""".strip()

no_rag_msg = llm.invoke([
    ("system", system_prompt),
    ("human", no_rag_prompt),
])

print("=== WITHOUT RAG ===")
print(no_rag_msg.content)

=== WITHOUT RAG ===
อาหารที่ควบคุมในการส่งออกไปจีน มีหลายประเภท แต่ตามข้อมูลที่มีอยู่ในปัจจุบัน (ซึ่งอาจมีการเปลี่ยนแปลง) อาหารที่ควบคุมหรือต้องมีการตรวจสอบอย่างเข้มงวดก่อนการส่งออกไปจีน ได้แก่:

1. **ผลิตภัณฑ์จากเนื้อสัตว์** - เช่น เนื้อวัว, เนื้อหมู, ไก่, และผลิตภัณฑ์จากสัตว์ปีกอื่นๆ
2. **ผลิตภัณฑ์จากปลาและผลิตภัณฑ์จากทะเล** - รวมถึงปลา, กุ้ง, และผลิตภัณฑ์จากสัตว์ทะเลอื่นๆ
3. **ผลิตภัณฑ์จากนม** - เช่น นม, ชีส, และผลิตภัณฑ์จากนมอื่นๆ
4. **ผลิตภัณฑ์จากพืช** - เช่น ข้าว, ข้าวโพด, และผลิตภัณฑ์จากพืชอื่นๆ ที่มีการควบคุม
5. **อาหารเสริม** - รวมถึงวิตามิน, แร่ธาตุ, และผลิตภัณฑ์อาหารเสริมอื่นๆ
6. **ผลิตภัณฑ์จากไข่** - รวมถึงไข่ไก่และผลิตภัณฑ์จากไข่อื่นๆ

นอกจากนี้ ยังมีอาหารอื่นๆ ที่ต้องมีการตรวจสอบและได้รับการอนุมัติก่อนการส่งออกไปจีน เช่น อาหารที่มีส่วนผสมของจีน, อาหารที่มีการใช้สารเคมีหรือสาร添加, และอาหารที่มีการผลิตหรือแปรรูปในประเทศที่มีการควบคุมอย่างเข้มงวด

ควรทราบว่าข้อมูลเหล่านี้อาจมีการเปลี่ยนแปลง และการควบคุมอาหารที่ส่งออกไปจีนอาจมีการปรับเปลี่ยนตามนโยบายและข้อกำหนดของจีน ดังนั้น จ

In [ ]:
def format_docs_for_prompt(retrieved_docs):
    formatted_docs = []
    for doc in retrieved_docs:
        title = doc.metadata.get("title", "Unknown Title")
        page = doc.metadata.get("page")

        # Add 1 to page number because it's usually 0-indexed in metadata
        page_str = f"p. {page + 1}" if page is not None else "Unknown Page"

        # Construct a citation label based on the document title
        citation_label = title
        if "TAS" in title:
            # For TAS documents, use the standard number as the label
            citation_label = title.split(":")[0].strip()
        elif "ACFS Manual" in title and "Decree 248" in title:
            # For the specific ACFS Manual, use the requested label
            citation_label = "ACFS Decree 248 Manual"

        formatted_docs.append(f"Document: [{citation_label}, {page_str}]\nContent:\n{doc.page_content}\n")
    return "\n---\n".join(formatted_docs)

These retrived docs will be paste into the prompt.

In [ ]:
print(format_docs_for_prompt(retrieved_docs[:4]))

Document: [ACFS Decree 248 Manual, p. 5]
Content:
ผู้ผลิตก่อน จึงจะสามารถส่งออกไปจีนได้ โดยมีผลบังคับใช้ตั้งแต่วันที่ 1 มกราคม 2565 เป็นต้นมา
โดยระเบียบดังกล่าวนำมาแทนที่กฎระเบียบ Decree 145 ว่าด้วยการขึ้นทะเบียน ผู้ผลิตอาหาร
ในต่างประเทศของสำนักงานควบคุมคุณภาพ ตรวจสอบและกักกันโรคแห่งประเทศจีน (The General Administration
of Quality Supervision, Inspection and Quarantine: AQSIQ Decree 145) ที่เดิมระบุ เฉพาะ สินค้าเสี่ยงสูง
บางรายการ (เนื้อสัตว์ ผลิตภัณฑ์นม สินค้าประมง และรังนก) ต้องขึ้นทะเบียนดังกล่าว
ในการขึ้นทะเบียนสำนักงานศุลกากรจีน (GACC) จะให้ดำเนินการผ่านระบบที่เรียกว่า “CIFER” ซึ่งจะ
ดำเนินการผ่าน...
CIFER.SINGLEWINDOW.CN

---
Document: [ACFS Decree 248 Manual, p. 6]
Content:
ระเบียบ 248 นี้ประกอบด้วย 4 บท 28 ข้อ ครอบคลุมเงื่อนไขและวิธีการการขึ้นทะเบียนของผู้ผลิต
อาหารส่งออกไปยังจีน มีรายละเอียดสรุปข้อสำคัญต่างๆ ดังนี้
บทที่ 1: บทบัญญัติทั่วไป
ข้อที่ 2 กำหนดนิยามของผู้ผลิตอาหารจากต่างประเทศ ซึ่งครอบคลุม ผู้ผลิต ผู้แปรรูป
และสถานที่เก็บสินค้าที่ผลิตอาหาร ส่งออกมายังจีน (ไม่รวมถึงผ

In [ ]:
context_text = format_docs_for_prompt(retrieved_docs)

rag_prompt = f"""
You are an assistant that answers questions from PDF documents related to food QA / regulatory topics.

Answer using only the information provided in the Retrieved PDF context.

Question:
{query}

Retrieved PDF context:
{context_text}

Answering rules:
- Answer in Thai.
- Use only information that appears in the retrieved context.
- If the information is not found in the context, answer: "ไม่พบข้อมูลนี้ในบริบทที่ดึงมา"
- Every bullet point must include a citation, for example: [ACFS Decree 248 Manual, p. 10]
""".strip()

rag_msg = llm.invoke([
    ("system", system_prompt),
    ("human", rag_prompt),
])

import textwrap

print("\n" + "=" * 90)
print("WITH RAG")
print("=" * 90 + "\n")

output = str(rag_msg.content).replace("\\n", "\n")

for line in output.split("\n"):
    if line.strip() == "":
        print()
    else:
        print(textwrap.fill(line, width=150))


WITH RAG

อาหารที่ควบคุมในการส่งออกไปจีน มี 18 ประเภท ดังนี้
* เนื้อสัตว์และผลิตภัณฑ์จากเนื้อสัตว์ [ACFS Decree 248 Manual, p. 10]
* ผลิตภัณฑ์ไส้บรรจุไส้กรอก [ACFS Decree 248 Manual, p. 10]
* รังนกและผลิตภัณฑ์รังนก [ACFS Decree 248 Manual, p. 10]
* ผลิตภัณฑ์สัตว์น้ำ [ACFS Decree 248 Manual, p. 10]
* ผลิตภัณฑ์จากผึ้ง [ACFS Decree 248 Manual, p. 10]
* ไข่และผลิตภัณฑ์จากไข่ [ACFS Decree 248 Manual, p. 10]
* ผลิตภัณฑ์นม [ACFS Decree 248 Manual, p. 10]
* Edible oil and oilseeds [ACFS Decree 248 Manual, p. 10]
* Stuffed pastry products [ACFS Decree 248 Manual, p. 10]
* เมล็ดธัญพืชเพื่อการบริโภค [ACFS Decree 248 Manual, p. 10]
* Grain milling industrial products and malt [ACFS Decree 248 Manual, p. 10]
* Fresh and dehydrated vegetables, dried beans [ACFS Decree 248 Manual, p. 10]
* Natural plant spices [ACFS Decree 248 Manual, p. 10]
* Nuts and seeds [ACFS Decree 248 Manual, p. 10]
* Dried fruits [ACFS Decree 248 Manual, p. 10]
* Unroasted coffee and cocoa beans [ACFS Decree 248 Manual, p. 1

## 7. Wrap-up

Across the lab:

- Specific PDF references -> LangChain documents (Step xxx)
- PDF pages -> retrievable chunks
- Chunks -> embeddings -> vector store
- Specific food-domain question -> retrieved PDF context
- No-RAG answer -> source-grounded RAG answer

**Key message:** RAG is useful when the AI needs to answer from a controlled knowledge base. For food workflows, RAG should support evidence review and source-grounded drafting, while final decisions stay with QA / regulatory / Food Safety owners.